<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/vector_stores/AzureCosmosDBNoSqlDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Azure Cosmos DB for NoSQL — Vector Store Demo

This notebook demonstrates how to use **AzureCosmosDBNoSqlVectorSearch** as a vector store in LlamaIndex.

It covers:
- Setting up the vector store with Azure OpenAI embeddings
- All supported search types: **vector**, **vector with score threshold**, **full text search**, **full text ranking**, **hybrid (RRF)**, **hybrid with score threshold**, and **weighted hybrid**
- Query options: WHERE filter, OFFSET/LIMIT pagination, projection mapping, and returning embeddings

If you're opening this notebook on Colab, install the required packages first.

## 1. Install dependencies

In [ ]:
%pip install llama-index
%pip install llama-index-vector-stores-azurecosmosnosql
%pip install llama-index-embeddings-azure-openai
%pip install llama-index-llms-azure-openai

## 2. Configure Azure OpenAI

Set your Azure OpenAI credentials. Replace the `YOUR_...` placeholders with your actual values.
The embedding model encodes documents and queries; the LLM generates answers.

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.llms.azure_openai import AzureOpenAI

aoai_api_key = "YOUR_AZURE_OPENAI_API_KEY"
aoai_endpoint = (
    "YOUR_AZURE_OPENAI_ENDPOINT"  # e.g. https://<resource>.openai.azure.com/
)
aoai_api_version = "2024-02-01"

llm = AzureOpenAI(
    model="YOUR_AZURE_OPENAI_COMPLETION_MODEL_NAME",
    deployment_name="YOUR_AZURE_OPENAI_COMPLETION_DEPLOYMENT_NAME",
    api_key=aoai_api_key,
    azure_endpoint=aoai_endpoint,
    api_version=aoai_api_version,
)

embed_model = AzureOpenAIEmbedding(
    model="YOUR_AZURE_OPENAI_EMBEDDING_MODEL_NAME",
    deployment_name="YOUR_AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME",
    api_key=aoai_api_key,
    azure_endpoint=aoai_endpoint,
    api_version=aoai_api_version,
)

Settings.llm = llm
Settings.embed_model = embed_model

## 3. Load documents

We use the Paul Graham essay included with LlamaIndex as sample data.

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["../data/paul_graham/paul_graham_essay.txt"]
).load_data()

print(
    f"Loaded {len(documents)} document(s). First doc ID: {documents[0].doc_id}"
)

## 4. Create the Azure Cosmos DB NoSQL vector store

### 4a. Define container policies

The **indexing policy** tells Cosmos DB to build a vector index on the `/embedding` path.  
The **vector embedding policy** describes the shape of the stored vectors.

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey

COSMOS_URI = "YOUR_AZURE_COSMOSDB_URI"  # e.g. https://<account>.documents.azure.com:443/
COSMOS_KEY = "YOUR_AZURE_COSMOSDB_KEY"

indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": '"_etag"/?'}],
    "vectorIndexes": [{"path": "/embedding", "type": "quantizedFlat"}],
}

vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/embedding",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 3072,
        }
    ]
}

partition_key = PartitionKey(path="/id")
cosmos_container_properties = {"partition_key": partition_key}
cosmos_database_properties = {}

### 4b. Initialise the vector store

You can create the store directly with a `CosmosClient`, or use one of the built-in factory methods:

```python
# From host + key
store = AzureCosmosDBNoSqlVectorSearch.from_host_and_key(host=COSMOS_URI, key=COSMOS_KEY, ...)

# From connection string
store = AzureCosmosDBNoSqlVectorSearch.from_connection_string(
    connection_string="AccountEndpoint=...;AccountKey=...;", ...
)

# From managed identity
store = AzureCosmosDBNoSqlVectorSearch.from_uri_and_managed_identity(
    cosmos_uri=COSMOS_URI, ...
)
```

Here we use a direct `CosmosClient`:

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.azurecosmosnosql import (
    AzureCosmosDBNoSqlVectorSearch,
)

client = CosmosClient(COSMOS_URI, credential=COSMOS_KEY)

vector_store = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client=client,
    vector_embedding_policy=vector_embedding_policy,
    indexing_policy=indexing_policy,
    cosmos_container_properties=cosmos_container_properties,
    cosmos_database_properties=cosmos_database_properties,
    database_name="llamaindex_db",
    container_name="llamaindex_container",
    create_container=True,
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)
print("Index created successfully.")

## 5. Standard query engine

Use the index as a standard LlamaIndex query engine. This performs vector search internally.

In [ ]:
import textwrap

query_engine = index.as_query_engine()
response = query_engine.query("What did the author love working on?")
print(textwrap.fill(str(response), 100))

## 6. Search types

All search types are available by passing `search_type` directly to `vector_store.query()`.

### 6a. Vector search

In [ ]:
from llama_index.core.vector_stores.types import VectorStoreQuery

query_embedding = embed_model.get_query_embedding(
    "What did the author work on?"
)

result = vector_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=3),
    search_type="vector",
)

print(f"Returned {len(result.nodes)} node(s)")
for i, (node, score) in enumerate(zip(result.nodes, result.similarities)):
    print(f"\n[{i + 1}] score={score:.4f}")
    print(textwrap.fill(node.get_content()[:200], 100))

### 6b. Vector search with score threshold

Only nodes whose similarity score exceeds `threshold` are returned.

In [ ]:
result = vector_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=10),
    search_type="vector_score_threshold",
    threshold=0.8,
)

print(f"Nodes above threshold: {len(result.nodes)}")
for node, score in zip(result.nodes, result.similarities):
    print(f"  score={score:.4f} | {node.get_content()[:80]}...")

### 6c. Full text search

Filters nodes using a `FullTextContains` predicate.

> **Prerequisites:** the container must be created with `full_text_search_enabled=True`, a `full_text_policy`, and `fullTextIndexes` in the indexing policy (see Section 8).

In [ ]:
# Requires a full-text-enabled store — see Section 8
result = fts_store.query(
    VectorStoreQuery(similarity_top_k=5),
    search_type="full_text_search",
    where="FullTextContains(c.text, 'Lisp')",
)

print(f"Full text matches: {len(result.nodes)}")
for node in result.nodes:
    print(f"  {node.get_content()[:100]}...")

### 6d. Full text ranking

Ranks nodes by `FullTextScore` using `ORDER BY RANK`. Multiple `full_text_rank_filter` entries are
fused with `ORDER BY RANK RRF(...)`.

In [ ]:
result = fts_store.query(
    VectorStoreQuery(similarity_top_k=5),
    search_type="full_text_ranking",
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "Lisp programming"},
    ],
)

print(f"Ranked results: {len(result.nodes)}")
for i, node in enumerate(result.nodes):
    print(f"  [{i + 1}] {node.get_content()[:100]}...")

### 6e. Hybrid search (RRF)

Fuses `FullTextScore` and `VectorDistance` rankings using Reciprocal Rank Fusion.

In [ ]:
result = fts_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=5),
    search_type="hybrid",
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "Lisp programming"},
    ],
)

print(f"Hybrid results: {len(result.nodes)}")
for i, node in enumerate(result.nodes):
    print(f"  [{i + 1}] {node.get_content()[:100]}...")

### 6f. Hybrid search with score threshold

Same as hybrid but applies a client-side score filter after retrieval.

In [ ]:
result = fts_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=10),
    search_type="hybrid_score_threshold",
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "Lisp programming"},
    ],
    threshold=0.5,
)

print(f"Hybrid results above threshold: {len(result.nodes)}")

### 6g. Weighted hybrid search

Runs the same RRF query as `hybrid` but applies **client-side per-component weights** when
re-ranking results.

`weights` length = number of `full_text_rank_filter` entries **+ 1** (the last entry is the weight
for the vector component).

> **Note:** The CosmosDB NoSQL API does not support a `weight=` parameter inside
> `ORDER BY RANK RRF(...)`, so re-ranking is performed client-side using position-based RRF scores.

In [ ]:
# 30 % text relevance, 70 % vector similarity
result = fts_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=5),
    search_type="weighted_hybrid_search",
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "Lisp programming"},
    ],
    weights=[0.3, 0.7],  # [text_weight, vector_weight]
)

print(f"Weighted hybrid results: {len(result.nodes)}")
for i, node in enumerate(result.nodes):
    print(f"  [{i + 1}] {node.get_content()[:100]}...")

In [ ]:
# Two full-text components + one vector component
result = fts_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=5),
    search_type="weighted_hybrid_search",
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "Lisp"},
        {"search_field": "text", "search_text": "programming language"},
    ],
    weights=[0.2, 0.3, 0.5],  # [text1_weight, text2_weight, vector_weight]
)

print(f"Multi-component weighted results: {len(result.nodes)}")

## 7. Query options

### 7a. WHERE filter

Apply any CosmosDB SQL predicate to restrict results.

In [ ]:
result = vector_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=5),
    search_type="vector",
    where="c.metadata.theme = 'programming'",
)

print(f"Filtered results: {len(result.nodes)}")

### 7b. Pagination (OFFSET / LIMIT)

In [ ]:
# Skip the first 5 results and return the next 3
result = vector_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=10),
    search_type="vector",
    offset_limit="OFFSET 5 LIMIT 3",
)

print(f"Paginated results (offset=5, limit=3): {len(result.nodes)}")

### 7c. Projection mapping

Return only specific document fields, surfaced as node metadata keys.

In [ ]:
result = vector_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=3),
    search_type="vector",
    projection_mapping={"id": "id", "text": "text"},
)

for node in result.nodes:
    print(node.metadata)  # Only 'id' and 'text' keys are present

### 7d. Return vector embeddings

In [ ]:
result = vector_store.query(
    VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=1),
    search_type="vector",
    return_with_vectors=True,
)

node = result.nodes[0]
embedding = node.metadata.get("vector", [])
print(f"Embedding dimensions: {len(embedding)}")

## 8. Full text search setup

To enable full text search or any hybrid search type, create the container with a
`full_text_policy` and include `fullTextIndexes` in the indexing policy.

In [ ]:
full_text_indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": '"_etag"/?'}],
    "vectorIndexes": [{"path": "/embedding", "type": "quantizedFlat"}],
    "fullTextIndexes": [{"path": "/text"}],
}

full_text_policy = {
    "defaultLanguage": "en-US",
    "fullTextPaths": [{"path": "/text", "language": "en-US"}],
}

fts_container_properties = {
    "partition_key": partition_key,
    "full_text_policy": full_text_policy,
}

fts_store = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client=client,
    vector_embedding_policy=vector_embedding_policy,
    indexing_policy=full_text_indexing_policy,
    cosmos_container_properties=fts_container_properties,
    cosmos_database_properties=cosmos_database_properties,
    database_name="llamaindex_db",
    container_name="llamaindex_fts_container",
    create_container=True,
    full_text_search_enabled=True,
)

fts_storage_context = StorageContext.from_defaults(vector_store=fts_store)
fts_index = VectorStoreIndex.from_documents(
    documents, storage_context=fts_storage_context
)
print("Full-text search index created successfully.")